# K-Means From Scratch

**What you'll build:** Full K-Means with random initialization and convergence check.

**Prerequisites:** NumPy, distance metrics, basic linear algebra.

## Concept Recap

**Objective (inertia):** minimize within-cluster sum of squared distances:
$$J = \sum_{k=1}^K \sum_{x \in C_k} \|x - \mu_k\|^2$$

**Algorithm (Lloyd's):**
1. **Initialize:** randomly pick k points as centroids
2. **Assignment:** assign each point to the nearest centroid
3. **Update:** move each centroid to the mean of its assigned points
4. **Repeat** until centroids stop moving (convergence)

**Convergence:** guaranteed to decrease J each iteration. Converges to a local minimum.
Run multiple times with different initializations; keep the run with lowest inertia.

**K-Means++:** instead of random init, choose initial centroids with probability proportional
to distance from already-chosen centroids. Dramatically improves convergence quality.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def kmeans(X, k, n_iter=100, tol=1e-4, random_state=42):
    np.random.seed(random_state)
    centroids = X[np.random.choice(len(X), k, replace=False)]
    for i in range(n_iter):
        dists = np.linalg.norm(X[:, None] - centroids[None], axis=2)  # (n, k)
        labels = np.argmin(dists, axis=1)
        new_centroids = np.array([X[labels == j].mean(axis=0) for j in range(k)])
        if np.linalg.norm(new_centroids - centroids) < tol:
            print(f"Converged at iteration {i + 1}")
            break
        centroids = new_centroids
    inertia = sum(np.sum((X[labels == j] - centroids[j])**2) for j in range(k))
    return labels, centroids, inertia

X = np.vstack([
    np.random.randn(100, 2) + [0, 0],
    np.random.randn(100, 2) + [5, 5],
    np.random.randn(100, 2) + [-3, 5]
])

labels, centers, inertia = kmeans(X, k=3)
print(f"Inertia: {inertia:.2f}")

In [ ]:
from sklearn.cluster import KMeans

sk_km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X)
print(f"sklearn inertia: {sk_km.inertia_:.2f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Cluster plot
colors = ['#e74c3c', '#2ecc71', '#3498db']
for j in range(3):
    ax1.scatter(X[labels==j, 0], X[labels==j, 1], c=colors[j], alpha=0.5, label=f'Cluster {j}')
ax1.scatter(centers[:, 0], centers[:, 1], c='black', marker='X', s=200, zorder=5, label='Centroids')
ax1.set_title('K-Means Clusters (k=3)'); ax1.legend()

# Elbow curve
inertias = [kmeans(X, k=k, random_state=42)[2] for k in range(1, 9)]
ax2.plot(range(1, 9), inertias, 'o-')
ax2.axvline(3, color='r', linestyle='--', label='k=3 (true)')
ax2.set_xlabel('k'); ax2.set_ylabel('Inertia')
ax2.set_title('Elbow Curve'); ax2.legend()

plt.tight_layout(); plt.show()

## Exercises

1. **K-Means++:** Implement the K-Means++ initialization strategy and compare convergence speed and final inertia vs random init.
2. **Silhouette score:** Implement silhouette score from scratch: $s = (b - a) / \max(a, b)$ where a=mean intra-cluster distance, b=mean nearest-cluster distance.
3. **DBSCAN comparison:** Use sklearn DBSCAN on a dataset with non-spherical clusters (moons or circles). Show where K-Means fails.

## Summary

- K-Means minimizes within-cluster inertia via alternating assignment/update steps
- Random init can get stuck in poor local minima — use K-Means++ or multiple restarts
- Elbow method + silhouette score help choose k

**Next:** [Neural Net From Scratch](neural-net-from-scratch.ipynb)